In [3]:
import os
import sys

%load_ext autoreload
%autoreload 2

# Trick to plot with tex
os.environ["LD_LIBRARY_PATH"] = ""
os.environ["CONDA_PREFIX"] = "/home/guerrini/.conda/envs/sp_validation_3.11"

sys.path.append(
    "/home/guerrini/sp_validation/cosmo_inference/scripts/"
)

from getdist import plots, loadMCSamples
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import chain_postprocessing as cp

plt.style.use(
    "/home/guerrini/matplotlib_config/paper.mplstyle"
)

plt.rc('text', usetex=True)

sns.set_palette("husl")

g = plots.get_subplot_plotter(width_inch=30)
g.settings.axes_fontsize=60
g.settings.axes_labelsize=60
g.settings.alpha_filled_add = 0.7
g.settings.legend_fontsize = 60

%matplotlib inline

#SPECIFY DATA DIRECTORY AND DESIRED CHAINS TO ANALYSE
root_dir = "/n09data/guerrini/output_chains/"
root_external = "/n09data/guerrini/output_chains/ext_data/"
blind = "B"

colour_blind = {
    "A": "royalblue",
    "B": "crimson",
    "C": "forestgreen"
}

roots = [
    f"SP_v1.4.6.3_leak_corr_{blind}",
    f"SP_v1.4.6.3_{blind}_fiducial_config",
    f"SP_v1.4.6.3_leak_corr_kmax=5Mpc_{blind}",
    f"SP_v1.4.6.3_leak_corr_kmax=3Mpc_{blind}",
    f"SP_v1.4.6.3_leak_corr_kmax=1Mpc_{blind}",
    f"SP_v1.4.6.3_leak_corr_include_large_scales_{blind}",
    f"SP_v1.4.6.3_leak_corr_small_scales_{blind}",
    f"SP_v1.4.6.3_leak_corr_large_scales_{blind}",
    f"SP_v1.4.6.3_leak_corr_halofit_{blind}",
    f"SP_v1.4.6.3_leak_corr_HMCode_nobar_{blind}",
    f"SP_v1.4.6.3_leak_corr_OneCov_{blind}",
    f"SP_v1.4.6.3_{blind}"
]

legend_labels = [
    r"UNIONS $C_\ell$ (This work)",
    r"UNIONS $\xi_\pm(\vartheta)$, (Goh et al., 2026)",
    r"$k_{\rm max}=5h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=2048$",
    r"$k_{\rm max}=3h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=1800$",
    r"$k_{\rm max}=1h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=500$",
    r"Include large scales, $\ell_{\rm max}=1600$",
    r"Small scales only",
    r"Large scales only",
    r"\texttt{Halofit}",
    r"\texttt{HMCode} no baryons",
    r"\texttt{OneCovariance} only",
    r"No leakage correction",
]

colours = [
    colour_blind["B"],
    "darkorange",
    "violet",
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
    colour_blind["B"],
]

categories = [
    "harmonic",
    "configuration",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic",
    "harmonic" 
]

# Add the blinds to each list
for bl in ["A", "B", "C"]:
    if bl != blind:
        roots.append(
            f"SP_v1.4.6.3_leak_corr_{bl}"
        )
        legend_labels.append(
            fr"UNIONS $C_\ell$, Blind {bl}"
        )
        colours.append(
            colour_blind[bl]
        )
        categories.append(
            "harmonic"
        )

print(roots)

['SP_v1.4.6.3_leak_corr_B', 'SP_v1.4.6.3_B_fiducial_config', 'SP_v1.4.6.3_leak_corr_kmax=5Mpc_B', 'SP_v1.4.6.3_leak_corr_kmax=3Mpc_B', 'SP_v1.4.6.3_leak_corr_kmax=1Mpc_B', 'SP_v1.4.6.3_leak_corr_include_large_scales_B', 'SP_v1.4.6.3_leak_corr_small_scales_B', 'SP_v1.4.6.3_leak_corr_large_scales_B', 'SP_v1.4.6.3_leak_corr_halofit_B', 'SP_v1.4.6.3_leak_corr_HMCode_nobar_B', 'SP_v1.4.6.3_leak_corr_OneCov_B', 'SP_v1.4.6.3_B', 'SP_v1.4.6.3_leak_corr_A', 'SP_v1.4.6.3_leak_corr_C']


In [4]:
chains = []
for i, root in enumerate(roots):
    category = categories[i]
    if category != "external":
        if category == 'configuration':
            path_samples = os.path.join(
                root_dir,
                f"{root}/samples_{root}.txt"
            )
            path_getdist = os.path.join(
                root_dir,
                f"{root}/getdist_{root}"
            )
        elif category == 'harmonic':
            path_samples = os.path.join(
                root_dir,
                f"{root}/{root}/samples_{root}_cell.txt"
            )
            path_getdist = os.path.join(
                root_dir,
                f"{root}/{root}/getdist_{root}"
            )
        elif category == "external_compute_sample":
            path_samples = os.path.join(
                root_dir,
                f"ext_data/{root}/samples_{root}.txt"
            )
            path_getdist = os.path.join(
                root_dir,
                f"ext_data/{root}/getdist_{root}"
            )
        else:
            raise ValueError(f"The category, {category}, of {root} is not correct")
        
        if category == "external_compute_sample" and "Legacy" in root:
            chain_type="nautilus"
        else:
            chain_type="polychord"
        cp.load_samples_and_write_paramnames(path_samples, path_getdist+".paramnames", chain_type=chain_type)
        cp.write_samples_getdist_format(path_samples, path_getdist+".txt", chain_type=chain_type)
        chains.append(
            cp.load_chain(path_getdist, smoothing_scale=0.5)
        )
    else:
        path_getdist = os.path.join(
            root_dir,
            f"ext_data/{root}/getdist_{root}"
        )
        chains.append(
            cp.load_chain(path_getdist)
        )

/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_B/SP_v1.4.6.3_leak_corr_B/getdist_SP_v1.4.6.3_leak_corr_B.txt
Removed no burn in
/n09data/guerrini/output_chains/SP_v1.4.6.3_B_fiducial_config/getdist_SP_v1.4.6.3_B_fiducial_config.txt
Removed no burn in


/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_kmax=5Mpc_B/SP_v1.4.6.3_leak_corr_kmax=5Mpc_B/getdist_SP_v1.4.6.3_leak_corr_kmax=5Mpc_B.txt
Removed no burn in
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_kmax=3Mpc_B/SP_v1.4.6.3_leak_corr_kmax=3Mpc_B/getdist_SP_v1.4.6.3_leak_corr_kmax=3Mpc_B.txt
Removed no burn in
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_kmax=1Mpc_B/SP_v1.4.6.3_leak_corr_kmax=1Mpc_B/getdist_SP_v1.4.6.3_leak_corr_kmax=1Mpc_B.txt
Removed no burn in


/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_include_large_scales_B/SP_v1.4.6.3_leak_corr_include_large_scales_B/getdist_SP_v1.4.6.3_leak_corr_include_large_scales_B.txt
Removed no burn in
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_small_scales_B/SP_v1.4.6.3_leak_corr_small_scales_B/getdist_SP_v1.4.6.3_leak_corr_small_scales_B.txt
Removed no burn in


/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_large_scales_B/SP_v1.4.6.3_leak_corr_large_scales_B/getdist_SP_v1.4.6.3_leak_corr_large_scales_B.txt
Removed no burn in
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_halofit_B/SP_v1.4.6.3_leak_corr_halofit_B/getdist_SP_v1.4.6.3_leak_corr_halofit_B.txt
Removed no burn in
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_HMCode_nobar_B/SP_v1.4.6.3_leak_corr_HMCode_nobar_B/getdist_SP_v1.4.6.3_leak_corr_HMCode_nobar_B.txt
Removed no burn in
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_OneCov_B/SP_v1.4.6.3_leak_corr_OneCov_B/getdist_SP_v1.4.6.3_leak_corr_OneCov_B.txt
Removed no burn in


/n09data/guerrini/output_chains/SP_v1.4.6.3_B/SP_v1.4.6.3_B/getdist_SP_v1.4.6.3_B.txt
Removed no burn in
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_A/SP_v1.4.6.3_leak_corr_A/getdist_SP_v1.4.6.3_leak_corr_A.txt


...

<Figure size 600x450 with 0 Axes>

<Figure size 600x450 with 0 Axes>

<Figure size 600x450 with 0 Axes>

<Figure size 600x450 with 0 Axes>

<Figure size 600x450 with 0 Axes>

In [5]:
name_list = ['OMEGA_M','ombh2','h0','n_s','SIGMA_8','S_8','s_8_input', 'logt_agn','a','m1','bias_1']
label_list = [r'\Omega_{\rm m}', r'\omega_b h^2', r'h_0', r'n_s', r'\sigma_8', r'S_8', r'S_8', r'\log T_{\rm AGN}', r'A_{\rm IA}', r'm_1', r'\Delta z_1']

for i, chain in enumerate(chains):
    print(legend_labels[i])
    param_names = chain.getParamNames()
    for name, label in zip(name_list, label_list):
        try:
            param_names.parWithName(name).label = label
        except:
            warnings.warn(f"Parameter {name} not found in chain {roots[i]}.")

UNIONS $C_\ell$ (This work)
UNIONS $\xi_\pm(\vartheta)$, (Goh et al., 2026)
$k_{\rm max}=5h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=2048$
$k_{\rm max}=3h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=1800$
$k_{\rm max}=1h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=500$
Include large scales, $\ell_{\rm max}=1600$
Small scales only
Large scales only
\texttt{Halofit}
\texttt{HMCode} no baryons
\texttt{OneCovariance} only
No leakage correction
UNIONS $C_\ell$, Blind A
UNIONS $C_\ell$, Blind C


In [6]:
best_fit_method = "2Dkde" 

param_values = np.array(["# Expt", "Colour", "S8_Mean", "S8_low", "S8_high",  "sigma_8_Mean", "sigma_8_low", "sigma_8_high", "Omega_m_Mean", "Omega_m_low", "Omega_m_high"])
escaped = np.char.replace(legend_labels, '\\', '\\\\')
best_fit = {}
for i, chain in enumerate(chains):
    print(chain.root)
    margestats = chain.getMargeStats()
    likestats = chain.getLikeStats()

    s8_stats = margestats.parWithName('S_8')
    sigma8_stats = margestats.parWithName('SIGMA_8')
    omegam_stats = margestats.parWithName('OMEGA_M')

    best_fit[roots[i]] = cp.extract_best_fit_params(chain, best_fit_method=best_fit_method)

    for param_name in name_list:
        upper, lower, _, _ = cp.compute_limits(chain, param_name)
        try:
            param_stats = margestats.parWithName(param_name)
            best_fit[roots[i]][f"{param_name}_low"] = lower
        
            best_fit[roots[i]][f"{param_name}_high"] = upper
        except:
            warnings.warn(f"Parameter {param_name} not found in chain {roots[i]}.")

/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_B/SP_v1.4.6.3_leak_corr_B/getdist_SP_v1.4.6.3_leak_corr_B
/n09data/guerrini/output_chains/SP_v1.4.6.3_B_fiducial_config/getdist_SP_v1.4.6.3_B_fiducial_config
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_kmax=5Mpc_B/SP_v1.4.6.3_leak_corr_kmax=5Mpc_B/getdist_SP_v1.4.6.3_leak_corr_kmax=5Mpc_B
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_kmax=3Mpc_B/SP_v1.4.6.3_leak_corr_kmax=3Mpc_B/getdist_SP_v1.4.6.3_leak_corr_kmax=3Mpc_B
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_kmax=1Mpc_B/SP_v1.4.6.3_leak_corr_kmax=1Mpc_B/getdist_SP_v1.4.6.3_leak_corr_kmax=1Mpc_B
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_include_large_scales_B/SP_v1.4.6.3_leak_corr_include_large_scales_B/getdist_SP_v1.4.6.3_leak_corr_include_large_scales_B
/n09data/guerrini/output_chains/SP_v1.4.6.3_leak_corr_small_scales_B/SP_v1.4.6.3_leak_corr_small_scales_B/getdist_SP_v1.4.6.3_leak_corr_small_scales_B
/n09data/guerrini/output_chains/SP_v

In [7]:
# Run the best fit estimation for all the chains
path_ini_files = '/home/guerrini/sp_validation/cosmo_inference/cosmosis_config'
fiducial_root_cell = fiducial_root_cell = f"SP_v1.4.6.3_leak_corr_{blind}"
fiducial_root_xi_data = f"SP_v1.4.6.3_leak_corr_{blind}_masked"
fiducial_root_xi_chains = f"SP_v1.4.6.3_{blind}_fiducial_config"

for i, root in enumerate(roots):
    print(root)
    if categories[i] == "harmonic":
        blind_ = root.split("_")[-1]
        cp.compute_best_fit(
            path_ini_files, 
            best_fit[root],
            root,
            is_harmonic=True,
            blind=blind_
        )
    elif categories[i] == "configuration":
        ini_file_root = os.path.join(
            path_ini_files,
            f'config_space_v1.4.6.3_fiducial/pipeline/blind_{blind}/fiducial.ini'
        )
        cp.compute_best_fit(
            path_ini_files,
            best_fit[root],
            fiducial_root_xi_chains,
            is_harmonic=False,
            blind=blind,
            ini_file_root=ini_file_root
        )

SP_v1.4.6.3_leak_corr_B
STDOUT:
Setting up pipeline from parameter file /home/guerrini/sp_validation/cosmo_inference/cosmosis_config/harmonic_space_fiducial_B/cosmosis_pipeline_SP_v1.4.6.3_leak_corr_B_cell.ini
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Setting up module consistency
------------------------------

Setting up module sample_S8
----------------------------

Setting up module camb
-----------------------
**** WARNING: Parameter 'kmin' in the [camb] section never used!


Setting up module load_nz_fits
-------------------------------
Loading number density data from data/SP_v1.4.6.3_leak_corr_B/cosmosis_SP_v1.4.6.3_leak_corr_B.fits:
    Looking at FITS extension NZ_SOURCE:
        Found 1 bins
        Putting n(0) = 0 at the start of the n(z)

Setting up module photoz_bias
------------------------------

Setting up module linear_alignment
---

In [12]:
from astropy.io import fits
from scipy.interpolate import interp1d

def get_chi2_ndata_xi_fiducial(output_chains):
    path_best_fit_output = f"{output_chains}/SP_v1.4.6.3_{blind}_fiducial_config/best_fit/"

    data = fits.open(
        f"/home/guerrini/sp_validation/cosmo_inference/data/SP_v1.4.6.3_config/SP_v1.4.6.3_{blind}/cosmosis_SP_v1.4.6.3_leak_corr_{blind}_masked.fits"
    )

    # Load the best fit data vector
    theta = np.loadtxt(
        path_best_fit_output + "/shear_xi_plus/theta.txt"
    )
    theta_arcmin = theta * 180 * 60 / np.pi
    shear_xi_plus = np.loadtxt(
        path_best_fit_output + "/shear_xi_plus/bin_1_1.txt"
    )
    shear_xi_minus = np.loadtxt(
        path_best_fit_output + "/shear_xi_minus/bin_1_1.txt"
    )
    xi_sys_plus = np.loadtxt(
        path_best_fit_output + "/xi_sys/shear_xi_plus.txt"
    )
    xi_sys_minus = np.loadtxt(
        path_best_fit_output + "/xi_sys/shear_xi_minus.txt"
    )

    # Load the data vectors and covariance matrix
    theta_data = data['XI_PLUS'].data['ANG']
    xi_plus_data = data['XI_PLUS'].data['VALUE']
    xi_minus_data = data['XI_MINUS'].data['VALUE']
    
    # interpolate the model
    interp_xi_plus = interp1d(theta_arcmin, shear_xi_plus, kind='cubic', fill_value='extrapolate')
    interp_xi_minus = interp1d(theta_arcmin, shear_xi_minus, kind='cubic', fill_value='extrapolate')

    xi_plus_model = interp_xi_plus(theta_data)
    xi_plus_model += xi_sys_plus
    xi_minus_model = interp_xi_minus(theta_data)
    xi_minus_model += xi_sys_minus


    cov = data['COVMAT'].data[0:2*len(theta_data), 0:2*len(theta_data)]

    # Concatenate the data and model vectors
    xi_data = np.concatenate([xi_plus_data, xi_minus_data])
    xi_model = np.concatenate([xi_plus_model, xi_minus_model])

    lower_bound_xi_p = 12
    upper_bound_xi_p = 83
    lower_bound_xi_m = 12
    upper_bound_xi_m = 83

    selection = np.concatenate([
        (theta_data > lower_bound_xi_p) & (theta_data < upper_bound_xi_p),
        (theta_data > lower_bound_xi_m) & (theta_data < upper_bound_xi_m)
    ])
    
    xi_data = xi_data[selection]
    xi_model = xi_model[selection]
    cov = cov[selection][:, selection]

    n_data = len(xi_data)
    chi2 = (xi_data - xi_model).T @ np.linalg.inv(cov) @ (xi_data - xi_model)

    return chi2, n_data



def get_latex_table(roots, best_fit, categories, labels):
    latex_lines = [
        r"\begin{tabular}{l|c|c|c|c|c|c|c}",
        r"\hline",
        r"\hline",
        r"Experiment name & $S_8$ & $\Omega_m$ & $\sigma_8$ & $A_\mathrm{IA}$ & $\log T_\mathrm{AGN}$ & $\chi^2$ & $N_{\rm data}$ \\ ",
        r"\hline"
    ]

    for i, (root, cat) in enumerate(zip(roots, categories)):
        label = labels[i]
        best_fit_vals = best_fit[root]

        # read the chi2 and p-value from the best fit output
        path_best_fit_output = f"/n09data/guerrini/output_chains/{root}/best_fit/"

        if cat == 'harmonic':
            with open(f'{path_best_fit_output}/data_vector/values.txt', 'r') as f:
                for line in f:
                    # Remove whitespace and split by '='
                    if '=' in line:
                        key, value = line.strip().split('=')
                        key = key.strip()
                        value = float(value.strip()) # Convert to number
                        
                        if key == '2pt_like_chi2':
                            chi2 = value
                        elif key == '2pt_like_n':
                            n_data = int(value) # n is usually an integer count
        elif cat == 'configuration':
            output_chains = "/n09data/guerrini/output_chains/"
            chi2, n_data = get_chi2_ndata_xi_fiducial(output_chains)
        else:
            raise ValueError(f"Category {cat} not recognized for root {root}.")


        if ('halofit' in root) or ('HMCode_nobar' in root):
            logt_agn_str = "N/A"
        else:
            logt_agn_str = f"${best_fit_vals['logt_agn']:.3f}^{{+{(best_fit_vals['logt_agn_high']-best_fit_vals['logt_agn']):.3f}}}_{{-{(best_fit_vals['logt_agn'] - best_fit_vals['logt_agn_low']) :.3f}}}$"
        line = (
            f"{label} & "
            rf" ${best_fit_vals['S_8']:.3f}^{{+{(best_fit_vals['S_8_high'] - best_fit_vals['S_8']):.3f}}}_{{-{(best_fit_vals['S_8'] - best_fit_vals['S_8_low']):.3f}}}$ & "
            rf" ${best_fit_vals['OMEGA_M']:.3f}^{{+{(best_fit_vals['OMEGA_M_high'] - best_fit_vals['OMEGA_M']):.3f}}}_{{-{(best_fit_vals['OMEGA_M'] - best_fit_vals['OMEGA_M_low']):.3f}}}$ & "
            rf" ${best_fit_vals['SIGMA_8']:.3f}^{{+{(best_fit_vals['SIGMA_8_high'] - best_fit_vals['SIGMA_8']):.3f}}}_{{-{(best_fit_vals['SIGMA_8'] - best_fit_vals['SIGMA_8_low']):.3f}}}$ & "
            rf" ${best_fit_vals['a']:.3f}^{{+{(best_fit_vals['a_high'] - best_fit_vals['a']):.3f}}}_{{-{(best_fit_vals['a'] - best_fit_vals['a_low']):.3f}}}$ & "
            rf" {logt_agn_str} & "
            f"{chi2:.2f} & {n_data} \\\\"
        )
        latex_lines.append(line)

    latex_lines.append(r"\hline")
    latex_lines.append(r"\end{tabular}")

    # Print LaTeX table
    print("\n".join(latex_lines))

In [13]:
get_latex_table(roots, best_fit, categories, legend_labels)

\begin{tabular}{l|c|c|c|c|c|c|c}
\hline
\hline
Experiment name & $S_8$ & $\Omega_m$ & $\sigma_8$ & $A_\mathrm{IA}$ & $\log T_\mathrm{AGN}$ & $\chi^2$ & $N_{\rm data}$ \\ 
\hline
UNIONS $C_\ell$ (This work) &  $0.917^{+0.077}_{-0.079}$ &  $0.221^{+0.175}_{-0.058}$ &  $0.875^{+0.282}_{-0.162}$ &  $1.054^{+0.618}_{-0.706}$ &  $7.837^{+0.074}_{-0.417}$ & 10.20 & 17 \\
UNIONS $\xi_\pm(\vartheta)$, (Goh et al., 2026) &  $0.858^{+0.081}_{-0.082}$ &  $0.267^{+0.141}_{-0.072}$ &  $0.809^{+0.225}_{-0.152}$ &  $0.932^{+0.794}_{-0.714}$ &  $7.833^{+0.071}_{-0.359}$ & 10.09 & 14 \\
$k_{\rm max}=5h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=2048$ &  $0.925^{+0.082}_{-0.092}$ &  $0.226^{+0.155}_{-0.071}$ &  $1.000^{+0.218}_{-0.264}$ &  $1.110^{+0.603}_{-0.778}$ &  $7.797^{+0.125}_{-0.301}$ & 11.23 & 21 \\
$k_{\rm max}=3h\,\mathrm{Mpc}^{-1},\ \ell_{\rm max}=1800$ &  $0.914^{+0.081}_{-0.072}$ &  $0.219^{+0.152}_{-0.069}$ &  $0.972^{+0.238}_{-0.229}$ &  $0.970^{+0.711}_{-0.671}$ &  $7.801^{+0.122}_{-0.307}$ & 

In [25]:
best_fit

{'SP_v1.4.6.3_leak_corr_B': {'omch2': 0.09620992213847757,
  'h0': 0.6929003435684542,
  'ombh2': 0.022166368226223226,
  'n_s': 0.9953157423517123,
  's_8_input': 0.9169519860054174,
  'logt_agn': 7.8369287302294195,
  'a': 1.0542669195153391,
  'm1': -0.05816430536286979,
  'bias_1': -0.03090182286512097,
  'OMEGA_LAMBDA': 0.7615874901806983,
  'S_8': 0.9169519860054174,
  'SIGMA_8': 0.8749939074573827,
  'OMEGA_M': 0.22111706327726893,
  'OMEGA_M_low': 0.16345655347909832,
  'OMEGA_M_high': 0.3965643955094385,
  'ombh2_low': 0.02164386689637142,
  'ombh2_high': 0.02272558733389502,
  'h0_low': 0.6663219306894427,
  'h0_high': 0.7748025918992061,
  'n_s_low': 0.8754795112501559,
  'n_s_high': 1.0431763263117209,
  'SIGMA_8_low': 0.7125628115768233,
  'SIGMA_8_high': 1.1569904402428195,
  'S_8_low': 0.8382529871176014,
  'S_8_high': 0.9942050338606128,
  's_8_input_low': 0.8382529871176017,
  's_8_input_high': 0.9942050338606129,
  'logt_agn_low': 7.420045376642155,
  'logt_agn_high':